# Quantum AI — v12: verified reachable-angle portfolio

**Full local score: 22.80996264**, up from our v11's **22.80391880**.
The new component is SomeJoint's public 889-qubit DQI, used unchanged.
It reduces measured DQI cost by about 0.0795% relative to v11; this is an
incremental public-component update, not a major original breakthrough.
The linked Alex Chilton notebook's current v54 measures **22.79549878**;
its historic `22-57` slug is not its latest result.

This is a small, independently verified improvement. It has not reached 23-24.
The change since v11 comes from SomeJoint's newer DQI circuit, credited below.
Our v11 chemistry coefficient update is retained.
Our selected change lowers the same fixed-word model's normalization from
**48.61375151 to 48.58975683**, using float64 joint coefficients and a gauge.
U, angle_words, tree_edges, brot, H1_I and Hsym_lower remain bit-identical.

The notebook checks all files, reproduces the selected coefficient update,
proves the official reachable ANGLE contract for every HMR outcome, runs all
three complete unchanged official verifiers, and then exports submission.zip.


## The host's target-register contract

[Tanuj Khattar's official clarification](https://www.kaggle.com/competitions/google-verifiable-quantum-circuit-compilation/discussion/753092) requires correctness on the
states reached in the specified forward and reverse traversals. Forward
ANGLE_k starts from batch k-1 (zero for k=0); reverse starts from batch k.
The current public loaders use the sign encoding bit7=bit6 on those states.

Our previous interpretation unnecessarily required arbitrary incoming words.
That stronger test does fail on these public loaders; this does not invalidate
them under the host's ruling. We explicitly make no arbitrary-target claim.
The algebraic audit below checks all 1220 addresses, both required starting
states, clean workspace, restored address and all 2^80 HMR outcomes per circuit.


In [ ]:
import glob, hashlib, io, json, lzma, math, os, shutil, subprocess, sys, time, zipfile
from pathlib import Path
import numpy as np

INPUT=Path(os.environ.get('QCC_INPUT_DIR','/kaggle/input'))
WORK=Path(os.environ.get('QCC_WORK_DIR','/kaggle/working'));WORK.mkdir(parents=True,exist_ok=True)
def sha(path):
    with Path(path).open('rb')as f:return hashlib.file_digest(f,'sha256').hexdigest()
matches=sorted(Path(p)for p in glob.glob(str(INPUT/'**/components_manifest_v12.json'),recursive=True))
assert len(matches)==1,'Attach the pinned v12 components dataset'
mp=matches[0];assert sha(mp)=='20d2fb4be380874909e249f2bb90db8edc685c190967f2d47a11636ce20a281f'
MANIFEST=json.loads(mp.read_text());DATA=mp.parent
options=[Path(os.environ.get('QCC_COMP_DIR','/nonexistent')),
 INPUT/'competitions/google-verifiable-quantum-circuit-compilation',
 INPUT/'google-verifiable-quantum-circuit-compilation']
COMP=next((p for p in options if(p/'kickmix').is_dir()and(p/'verifiers').is_dir()),None)
assert COMP is not None,'Attach the competition data'
env=dict(os.environ,QCC_COMP_DIR=str(COMP),OMP_NUM_THREADS='1',OPENBLAS_NUM_THREADS='1',MKL_NUM_THREADS='1')
env['PYTHONPATH']=os.pathsep.join([str(COMP/'kickmix'),str(COMP/'verifiers'),str(COMP/'verifiers/dqi')])
SUB=WORK/'submission_v12';SUB.mkdir(exist_ok=True)
for item in MANIFEST['files']:
    src=DATA/item['stored_as'];assert sha(src)==item['stored_sha256']
    dst=SUB/item['file'];dst.parent.mkdir(parents=True,exist_ok=True)
    if item['xz']:
        with lzma.open(src,'rb')as fi,dst.open('wb')as fo:shutil.copyfileobj(fi,fo)
    else:shutil.copyfile(src,dst)
    assert sha(dst)==item['sha256']
bundle=DATA/MANIFEST['source_bundle']['stored_as'];assert sha(bundle)==MANIFEST['source_bundle']['sha256']
SOURCE=WORK/'v12_source';SOURCE.mkdir(exist_ok=True)
with lzma.open(bundle,'rb')as f,zipfile.ZipFile(io.BytesIO(f.read()))as z:z.extractall(SOURCE)
for name,pin in json.loads((SOURCE/'source_manifest.json').read_text()).items():assert sha(SOURCE/name)==pin,name
print('Authenticated 18 submission files and whitelisted reproduction sources.')


## Reproduce and check the coefficient update

Only W and H2_I change. The pinned public Gluzdov factor plus the selected
production delta reproduces those arrays on CPU. Optional CUDA search source
and numerical records are also available in v12_source/; this run needs no GPU.


In [ ]:
reproduced=WORK/'reproduced_coefficients.npz'
subprocess.run([sys.executable,str(SOURCE/'our/apply_weight_delta.py'),
 str(SOURCE/'public/gluzdov_original.npz'),str(SOURCE/'our/selected_weight_delta.npz'),str(reproduced)],env=env,check=True)
with np.load(reproduced,allow_pickle=False)as got,np.load(SUB/'dfthcsos/submission.npz',allow_pickle=False)as expected:
    assert set(got.files)==set(expected.files)
    for key in got.files:
        if key in ('W','H2_I'):assert np.allclose(got[key],expected[key],rtol=0,atol=1e-11),key
        else:assert got[key].dtype==expected[key].dtype and got[key].tobytes()==expected[key].tobytes(),key
    for key,pin in MANIFEST['immutable_arrays'].items():assert hashlib.sha256(expected[key].tobytes()).hexdigest()==pin,key
print('Coefficient reproduction and all six immutable-array checks PASS.')


## Exact reachable-state phase check

The bit-packed evaluator is adapted from mrmeows' public audit. Each address
is evaluated exactly; every HMR has its own symbolic phase coefficient. All
coefficients must be zero. The proof checks actual circuit text. A missing-CZ
mutation must fail as a negative control. This is an independent encoder of
the allowed gate semantics, supplemented by the official native simulation.


In [ ]:
proof_path=WORK/'reachable_proof.json'
subprocess.run([sys.executable,str(SOURCE/'our/verify_reachable.py'),str(SUB/'dfthcsos'),str(proof_path)],env=env,check=True)
proof=json.loads(proof_path.read_text());assert proof['status']=='PASS' and len(proof['circuits'])==15
subprocess.run([sys.executable,str(SOURCE/'our/proof_negative_control.py'),str(SUB/'dfthcsos')],env=env,check=True)


## Run the three complete official verifiers

Fresh isolated subprocesses, unchanged verifier files and full default shot
counts: 200,540,928 inverse-square-root shots, 102,400 DQI shots and 78,080
chemistry shots. The declared salt is notebook-verification. DQI failures,
if observed, remain in its repetition penalty and receipt; no salt retries.


In [ ]:
results={};started=time.time()
for task in ('invsqrt','dqi','dfthcsos'):
    path=SUB/('dfthcsos'if task=='dfthcsos'else task+'.kmb')
    report=WORK/(task+'_official.json')
    subprocess.run([sys.executable,str(SOURCE/'our/verify_task.py'),task,str(path),str(report),'--workers','2'],env=env,check=True)
    result=json.loads(report.read_text());assert result['status']=='PASS';results[task]=result
ratios={task:MANIFEST['reference_costs'][task]/r['score']for task,r in results.items()}
gm=math.prod(ratios.values())**(1/3)
print('Fresh official results:',json.dumps({t:{'status':r['status'],'cost':r['score'],'Q':r['num_qubits']}for t,r in results.items()}))
print(f'Local geometric mean: {gm:.8f}; elapsed {time.time()-started:.1f}s')
print('Control Alex v54:',MANIFEST['alex_v54_control']['geometric_mean'],'; our previous v11:',MANIFEST['previous_our_v11'])
assert gm>MANIFEST['previous_our_v11'],'Candidate did not improve on the pinned v11 result'


## Export the verified files

The submission contains exactly the 18 authenticated production members.
Every compressed member is read back and hashed. The verification receipt
and reachable-state certificate are separate outputs. Running this notebook
does not make a leaderboard submission.


In [ ]:
archive=WORK/'submission.zip'
with zipfile.ZipFile(archive,'w',zipfile.ZIP_DEFLATED,compresslevel=6)as z:
    for item in MANIFEST['files']:z.write(SUB/item['file'],item['file'])
with zipfile.ZipFile(archive)as z:
    assert len(z.namelist())==18 and set(z.namelist())=={i['file']for i in MANIFEST['files']}
    assert z.testzip()is None
    for item in MANIFEST['files']:
        with z.open(item['file'])as f:assert hashlib.file_digest(f,'sha256').hexdigest()==item['sha256']
receipt=dict(release=MANIFEST['release'],salt=MANIFEST['salt'],local_geometric_mean=gm,
 results=results,ratios=ratios,archive_sha256=sha(archive),archive_bytes=archive.stat().st_size,
 files=MANIFEST['files'],reachable_proof='reachable_proof.json',leaderboard_submission=False)
(WORK/'verification_v12.json').write_text(json.dumps(receipt,indent=2))
print('Wrote verified 18-member submission.zip; SHA-256',sha(archive))
print('All work complete. This is a local verifier score, not a new leaderboard claim.')


## Sources and attribution

# Sources and attribution

- **Starting portfolio:** Alex Chilton, v54 of
  https://www.kaggle.com/code/alexchilton/22-57-lb-verified-three-task-portfolio
  Independently rechecked here: 22.79549878 with our reference costs and declared
  salt. His notebook's historic slug is not its current score.
- **Inverse square root:** busyaprime/Akmal Xodarev's updated 63-qubit circuit,
  unchanged, from Alex's public output. Original:
  https://www.kaggle.com/code/busyaprime/lb-22-72431-22x-cheaper-than-google-s-circuits
  https://www.kaggle.com/datasets/busyaprime/33-qcc-v8-members
  His two rebuilt regions are on mrmeows' 63-qubit hybrid circuit; earlier
  lineage includes Phuc Thai/DePie, Dmitrii Gluzdov and marsyas. Apache-2.0.
- **Selected DQI, new in v12:** SomeJoint/Agniv Banerjee's 889-qubit decoder,
  unchanged, from the public output and source description:
  https://www.kaggle.com/code/somejoint/quantum-circuits-22-77-local-gm-889-qubit-dqi
  Their paired high-component charts, inverse19 reverse, streamed synthesis
  divide and retimed heads are their work. This replaces v11's Mudit
  Chaturvedi/muchat007 895-qubit decoder:
  https://www.kaggle.com/code/muchat007/895q-streamed-decoder-measured-node-choices
  On SomeJoint's streamed conversion/replay and their team's arithmetic.
  Their credited lineage includes GPT-6 Astra Pro, Ahmed Berat Ozer's v9
  narrowing, Alex Chilton's Schur design on jday96314's QuantumAI,
  mrmeows/Ozer/manwithacat mask projection, Dmitrii Gluzdov, ViberTeam and
  Phuc Thai. The original jday96314 repository has no explicit license; we do
  not redistribute its source or assert a blanket license over that history.
  Only the already-public compiled circuit is packaged here.
- **Chemistry factorization and words:** SomeJoint's public R8/B143/C60 model:
  https://www.kaggle.com/datasets/somejoint/qcc-our-chemistry-b143-c60-xor2
  Lineage: Michael Rustad, Ahmed Berat Ozer, busyaprime, lightningv08/Vladislav
  Lavrentev and mrmeows. The public starting coefficient file is Dmitrii
  Gluzdov's 29 October output, obtained through Alex's portfolio:
  https://www.kaggle.com/code/dmitriigluzdov/quantum-circuit-compilation-from-source
  Its angle_words, U, tree_edges, brot, H1_I and Hsym_lower are unchanged.
- **All 15 ANGLE circuits:** Heliosli's v31_mc14 dataset release, unchanged:
  https://www.kaggle.com/datasets/heliosli/qcc-circuits-live
  https://www.kaggle.com/code/heliosli/the-dataset-is-public-now-enjoy
  The compiler design is on SomeJoint's chemistry and kernel relations. We
  did not synthesize these production circuit binaries. Their exact emitter
  source was not included in that release; this notebook authenticates and
  verifies them, and does not claim a byte-for-byte source rebuild.
- **Our selected model update:** Ahmed Berat Ozer, float64 constrained joint
  coefficient fitting, implicit elimination of the gap scale, an augmented
  coefficient rotation and selected production W/H2_I delta. The augmented
  Gram gauge formulation is credited to Kirill Tymchishin and mrmeows:
  https://www.kaggle.com/code/kirilltym/quantum-circuits-exact-gauge-5-word-qrom
  Optional CUDA search sources and native numerical records are included.
- **Reachable-state phase certificate:** adapted from mrmeows' Apache-2.0
  bit-packed audit_affine_hmr.py in the public cost-first source package:
  https://www.kaggle.com/datasets/mrmeows/qcc-1375-open-circuits
  Our adaptation initializes the two official traversal domains and binds the
  certificates to the selected binaries. It checks every valid address and
  every independent HMR outcome. It is not an arbitrary-target XOR proof.
- **Official ANGLE ruling:** Tanuj Khattar, Competition Host:
  https://www.kaggle.com/competitions/google-verifiable-quantum-circuit-compilation/discussion/753092
  The circuits need only work on the reachable forward/reverse word states.
- **Organizers:** Google Quantum AI / Qualtran, task definitions, Kickmix,
  reference Hamiltonian and unchanged official verifiers.

Our scripts are Apache-2.0. Each upstream component retains its own terms.
No ownership of upstream algorithms, binaries or competition data is claimed.
